# From interference to a canonical Euclidean rhythm

## A proof-free visual guide to the Quplets paper

The paper asks a dynamical question: **if two regular cyclic subdivisions interfere, can one follow a distinguished family of crests until it becomes a canonical discrete rhythm?**

This notebook follows the paper's line of thought without reproducing its proofs. The numerical plots illustrate the statements; the paper supplies the rigorous continuation, ranking, and arithmetic arguments.

The route is:

1. mix two regular cosine grids continuously;
2. follow the $n$ crests anchored at the initial $n$-grid;
3. understand the exactly solvable balanced slice $k_c$;
4. replace crest positions by circular spacings—the **Huplet**;
5. identify its quantized endpoint—the **Quplet**—as a canonical Euclidean rhythm representative;
6. verify that the tracked crests remain the $n$ most prominent ones;
7. see why even $n$ produces two reflected answers through a pitchfork.

In [102]:
from io import BytesIO
from math import gcd
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq
from scipy.signal import argrelextrema

from IPython.display import display

try:
    from ipywidgets import FloatSlider, HTML, Image as WidgetImage, IntSlider, VBox
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Set this to True to regenerate the five static SVG files used by main.typ.
EXPORT_FIGURES = False

# Find the repository root even when the notebook is opened from python/notebooks/.
working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_directory, *working_directory.parents) if (path / "main.typ").exists()),
    working_directory,
)
FIGURES_DIR = PROJECT_ROOT / "Figures"

def maybe_save(fig, filename):
    if EXPORT_FIGURES:
        FIGURES_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURES_DIR / filename, format="svg", bbox_inches="tight", pad_inches=0.05)

print(f"Setup complete. Project root: {PROJECT_ROOT}")
print("Run the cells from top to bottom; each example displays beneath its code cell.")

Setup complete. Project root: /Users/malcolmbraff/Documents/GitHub/Quplets
Run the cells from top to bottom; each example displays beneath its code cell.


## 1. The deformation: two regular grids in one profile

Let $x$ be phase on the unit circle and let $1<n<d$ be coprime integers. The model is

$$F_k(x)=(1-k)\cos(2\pi n x)+k\cos(2\pi d x), \qquad 0\leq k\leq1.$$

At $k=0$, the profile is the regular $n$-fold wave and has crests at $i/n$. At $k=1$, it is the regular $d$-fold wave and has crests at $j/d$. Between them, crests move and additional stationary pairs can be born.

A phase is stationary when $G_k(x)=0$, where

$$G_k(x)=(1-k)n\sin(2\pi n x)+kd\sin(2\pi d x).$$

The factor $-2\pi$ from $F'_k$ is omitted. The paper's central objects are the $n$ **anchored crest branches** $X_i(k)$ that begin at $X_i(0)=i/n$.

In [103]:
def F(x, k, n, d):
    """The two-frequency cyclic profile F_k(x)."""
    return (1.0 - k) * np.cos(2 * np.pi * n * x) + k * np.cos(2 * np.pi * d * x)


def G(x, k, n, d):
    """Stationarity equation, with the common factor -2π removed."""
    return (1.0 - k) * n * np.sin(2 * np.pi * n * x) + k * d * np.sin(2 * np.pi * d * x)


def crest_indicator(x, k, n, d):
    """Positive exactly when the stationary point is a crest."""
    return (1.0 - k) * n**2 * np.cos(2 * np.pi * n * x) + k * d**2 * np.cos(2 * np.pi * d * x)


def nearest_integer(z):
    """Paper convention: choose the upper integer at a half-integer tie."""
    return int(np.floor(z + 0.5))


def parameter_grid(*special_values, points=501):
    """A uniform [0,1] grid containing the requested exact parameters."""
    return np.unique(np.r_[np.linspace(0.0, 1.0, points), special_values])


def anchored_position(n, d, anchor_index, k):
    """Numerically solve one non-tied branch in its nearest-grid corridor."""
    anchor = anchor_index / n
    endpoint = nearest_integer(d * anchor_index / n) / d

    if np.isclose(anchor, endpoint, atol=1e-15):
        return anchor
    if k <= 0.0:
        return anchor
    if k >= 1.0:
        return endpoint

    left, right = sorted((anchor, endpoint))
    return brentq(G, left, right, args=(k, n, d), xtol=2e-14, rtol=1e-14)


def anchored_branches(n, d, k_values, *, skip_indices=()):
    """Return lifted branches i=0,…,n; 0 and n represent one circular branch."""
    if not (1 < n < d) or gcd(n, d) != 1:
        raise ValueError("Expected coprime integers satisfying 1 < n < d.")

    skipped = set(skip_indices)
    positions = np.full((len(k_values), n + 1), np.nan)
    for i in range(n + 1):
        if i in skipped:
            continue
        positions[:, i] = [anchored_position(n, d, i, k) for k in k_values]
    return positions


def fraction_ticks(denominator):
    values = np.linspace(0.0, 1.0, denominator + 1)
    labels = [r"$0$"] + [rf"$\frac{{{i}}}{{{denominator}}}$" for i in range(1, denominator)] + [r"$1$"]
    return values, labels


def figure_to_png(fig):
    """Render one Matplotlib figure for an in-place notebook image widget."""
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    return buffer.getvalue()


def validate_pair(n, d, *, parity="odd"):
    if not 1 < n < d:
        return "Choose integers satisfying 1 &lt; n &lt; d."
    if gcd(n, d) != 1:
        return f"Choose a coprime pair; gcd({n},{d})={gcd(n, d)}."
    if parity == "odd" and n % 2 == 0:
        return "This figure follows the paper's odd-n branch; choose odd n."
    if parity == "even" and n % 2 != 0:
        return "The pitchfork requires even n (and coprimality then makes d odd)."
    return None


def parameterized_plot(builder, *, default_n, default_d, parity="odd", n_max=61, d_max=89, n_step=1):
    """Attach n,d controls to a builder returning (figure, state)."""
    state = {}
    if not WIDGETS_AVAILABLE:
        fig, payload = builder(default_n, default_d)
        state.update(payload)
        plt.show()
        return {"state": state, "n_slider": None, "d_slider": None}

    n_slider = IntSlider(value=default_n, min=2, max=n_max, step=n_step, description="n", continuous_update=False, layout={"width": "75%"})
    d_slider = IntSlider(value=default_d, min=3, max=d_max, step=1, description="d", continuous_update=False, layout={"width": "75%"})
    status = HTML()
    image = WidgetImage(format="png")

    def refresh(change=None):
        n, d = n_slider.value, d_slider.value
        error = validate_pair(n, d, parity=parity)
        if error:
            status.value = f"<b>{error}</b>"
            image.value = b""
            state.clear()
            return
        status.value = f"Computing (n,d)=({n},{d})…"
        fig, payload = builder(n, d)
        image.value = figure_to_png(fig)
        state.clear()
        state.update(payload)
        status.value = f"Current plot: (n,d)=({n},{d})."

    for control in (n_slider, d_slider):
        control.observe(refresh, names="value")
    refresh()
    display(VBox([n_slider, d_slider, status, image]))
    return {"state": state, "n_slider": n_slider, "d_slider": d_slider, "refresh": refresh}

print("Numerical model and continuation helpers loaded.")

Numerical model and continuation helpers loaded.


### Opening experiment: move $k$

This is the notebook version of [`slider_k.py`](../scripts/slider_k.py). It opens at the script's example $(n,d)=(5, 7)$, but $n$ and $d$ are now controls rather than fixed constants. Choose a coprime pair with $1<n<d$, then move $k$ from the broad $n$-fold profile toward the fine $d$-fold profile. The red points are the $n$ highest sampled crests.

Things to notice:

- the profile can acquire many more than seven local maxima;
- the selected high crests move continuously rather than jumping between arbitrary peaks;
- near $k=1$, they approach $n$ distinguished sites of the $d$-grid.

This visual experiment motivates the paper's three main questions: do the initial crests continue globally, are they always the highest crests, and what discrete pattern do they select?

In [104]:
DEFAULT_N, DEFAULT_D = 5, 7
slider_x = np.linspace(0.0, 1.0, 10_000, endpoint=False)

def plot_dynamic_profile(n, d, k):
    y = F(slider_x, k, n, d)
    peak_indices = argrelextrema(y, np.greater, mode="wrap")[0]
    selected = peak_indices[np.argsort(y[peak_indices])[-n:]]
    selected = np.sort(selected)
    peak_x = slider_x[selected]
    peak_y = y[selected]

    # Repeat x=0 at x=1 when it is selected, closing the displayed cycle.
    if np.any(np.isclose(peak_x, 0.0)):
        peak_x = np.r_[peak_x, 1.0]
        peak_y = np.r_[peak_y, peak_y[np.argmin(np.abs(peak_x[:-1]))]]

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(slider_x, y, color="tab:blue", linewidth=1.4)
    ax.scatter(peak_x, peak_y, color="tab:red", s=28, zorder=3)
    for phase in peak_x:
        ax.axvline(phase, color="0.75", linestyle="--", linewidth=0.7, zorder=0)
        if n <= 12:
            ax.text(phase, -1.07, f"{phase:.3f}", ha="center", va="top", fontsize=7)
    ax.set(xlim=(0, 1), ylim=(-1.18, 1.08), xlabel=r"$x$", ylabel=r"$F_k(x)$")
    ax.set_title(rf"Interference profile: $(n,d)=({n},{d})$, $k={k:.3f}$, $k_c={n/(n+d):.3f}$")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    return fig

if WIDGETS_AVAILABLE:
    n_slider = IntSlider(
        value=DEFAULT_N, min=2, max=20, step=1,
        description="n", continuous_update=False,
        layout={"width": "75%"},
    )
    d_slider = IntSlider(
        value=DEFAULT_D, min=3, max=80, step=1,
        description="d", continuous_update=False,
        layout={"width": "75%"},
    )
    k_slider = FloatSlider(
        value=DEFAULT_N / (DEFAULT_N + DEFAULT_D),
        min=0.0, max=0.999, step=0.001,
        description="k", continuous_update=True,
        readout_format=".3f", layout={"width": "75%"},
    )
    slider_status = HTML()
    slider_image = WidgetImage(format="png")

    def update_slider_image(change=None):
        n, d, k = n_slider.value, d_slider.value, k_slider.value
        if not 1 < n < d:
            slider_status.value = "<b>Choose integers satisfying 1 &lt; n &lt; d.</b>"
            slider_image.value = b""
            return
        repeated = gcd(n, d)
        if repeated == 1:
            slider_status.value = f"Current pair: (n,d)=({n},{d}); balanced value k<sub>c</sub>={n/(n+d):.4f}."
        else:
            slider_status.value = f"<b>Non-coprime pair:</b> gcd({n},{d})={repeated}; the primitive profile repeats {repeated} times."
        fig = plot_dynamic_profile(n, d, k)
        slider_image.value = figure_to_png(fig)

    for control in (n_slider, d_slider, k_slider):
        control.observe(update_slider_image, names="value")
    update_slider_image()
    display(VBox([n_slider, d_slider, k_slider, slider_status, slider_image]))
else:
    print("ipywidgets is unavailable; showing the balanced slice as a static fallback.")
    plot_dynamic_profile(DEFAULT_N, DEFAULT_D, DEFAULT_N / (DEFAULT_N + DEFAULT_D))
    plt.show()

## 2. Global continuation selects nearest-grid endpoints

For odd $n$, the paper proves that every initial crest continues uniquely for the whole deformation. The branches remain distinct nondegenerate maxima, preserve their circular order, and stay inside disjoint short corridors. Their endpoints are simply the nearest $d$-grid sites:

$$X_i(1)=\frac{1}{d}\left\lfloor\frac{di}{n}+\frac12\right\rfloor.$$

The controls below open at the paper's example $(n,d)=(5,7)$. For any valid odd pair, the left ticks are the starting $n$-grid and the right ticks are the target $d$-grid. The curves at $0$ and $1$ are two lifts of the same fixed circular branch.

In [105]:
def make_crest_trajectory_figure(n, d):
    k_values = parameter_grid(points=501)
    positions = anchored_branches(n, d, k_values)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.0, 1.0, n + 1))
    for i in range(n + 1):
        ax.plot(k_values, positions[:, i], color=colors[i], linewidth=1.6)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=r"$k$", ylabel=r"$X_i$", title=rf"$X_{{i,{n},{d}}}(k)$")
    ax.set_xticks([0.0, 1.0], [r"$0$", r"$1$"])
    left_ticks, left_labels = fraction_ticks(n)
    ax.set_yticks(left_ticks, left_labels)
    right_ax = ax.secondary_yaxis("right")
    right_ticks, right_labels = fraction_ticks(d)
    right_ax.set_yticks(right_ticks, right_labels)
    ax.tick_params(axis="both", labelsize=12)
    right_ax.tick_params(axis="y", labelsize=12)
    fig.tight_layout()
    if EXPORT_FIGURES:
        maybe_save(fig, "Crests.svg" if (n, d) == (5, 7) else f"Crests-n{n}-d{d}.svg")
    return fig, {"n": n, "d": d, "k_values": k_values, "positions": positions}

crest_controls = parameterized_plot(
    make_crest_trajectory_figure, default_n=5, default_d=7, parity="odd", n_max=19, d_max=41
)

The dynamical problem now has an arithmetic endpoint: nearest rounding assigns one $d$-grid site to each labeled starting crest. But this alone does not yet explain why a Euclidean rhythm appears, nor why these branches should be preferred when extra maxima exist. The next steps address those two questions.

## 3. The balanced slice reveals an exact intermediate grid

The derivative contributions have equal weight when

$$kd=(1-k)n, \qquad k_c=\frac{n}{n+d}.$$

At this value, the stationarity equation factors as

$$\sin(d\theta)+\sin(n\theta)=2\sin\!\left(\frac{(n+d)\theta}{2}\right)\cos\!\left(\frac{(d-n)\theta}{2}\right)=0.$$

This exposes a resonant $(n+d)$-grid family and a complementary family. The paper compares their heights and shows that the $n$ highest threshold crests are the resonant sites selected by the anchored branches.

In [106]:
def make_threshold_figure(n, d):
    N = n + d
    k_c = n / N
    x_values = np.linspace(0.0, 1.0, 4001)
    profile = F(x_values, k_c, n, d)
    selected_indices = np.array([nearest_integer(i * N / n) for i in range(n + 1)])
    selected_x = selected_indices / N
    fig, ax = plt.subplots(figsize=(10, 5))
    for j in range(N + 1):
        grid_x = j / N
        ax.axvline(grid_x, color="0.86", linestyle="--", linewidth=0.8, zorder=0)
        if N <= 25:
            ax.text(grid_x, -0.04, rf"$\frac{{{j}}}{{{N}}}$", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=8, clip_on=False)
    ax.plot(x_values, profile, color="tab:blue", linewidth=1.6)
    ax.scatter(selected_x, F(selected_x, k_c, n, d), color="tab:red", s=30, zorder=3)
    ax.set(xlim=(0, 1), xlabel=r"$x$", ylabel=r"$F_{k_c}(x)$", title=rf"Balanced threshold: $(n,d)=({n},{d})$, $k_c=\frac{{{n}}}{{{N}}}$")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    if EXPORT_FIGURES:
        maybe_save(fig, "Fk(x).svg" if (n, d) == (5, 8) else f"Fk-n{n}-d{d}.svg")
    return fig, {"n": n, "d": d, "N": N, "k_c": k_c, "selected_x": selected_x}

threshold_controls = parameterized_plot(
    make_threshold_figure, default_n=5, default_d=8, parity="odd", n_max=19, d_max=41
)

The controls open at $(n,d)=(5,8)$, where the red crests lie on the $13$-grid. For every valid selected pair, they lie on the corresponding $(n+d)$-grid. This is not just a convenient snapshot: after converting positions to spacings, the threshold configuration is exactly the endpoint Quplet for the enlarged grid size $n+d$.

## 4. From moving positions to a rhythm: the Huplet

Rhythm depends on gaps rather than absolute phases. With ordered lifts

$$0=X_0(k)<X_1(k)<\cdots<X_{n-1}(k)<1, \qquad X_n(k)=1,$$

define

$$D_i(k)=X_{i+1}(k)-X_i(k), \qquad \operatorname{Huplet}(n,d,k)=(D_0(k),\ldots,D_{n-1}(k)).$$

The Huplet is a continuous path of positive spacing vectors whose entries always sum to one. Most parameter values give irrational timing patterns. Three values are known to lock to finite grids: $k=0$, $k=k_c$, and $k=1$. The controls open at the paper's denser example $(53,67)$ and may take a moment to recompute for large values.

In [107]:
def make_spacing_figure(n, d):
    k_c = n / (n + d)
    k_values = parameter_grid(k_c, points=401)
    positions = anchored_branches(n, d, k_values)
    spacings = np.diff(positions, axis=1)
    fig, ax = plt.subplots(figsize=(10, 6))
    displayed_count = n // 2 + 1
    colors = plt.cm.viridis(np.linspace(0.0, 1.0, displayed_count))
    for i in range(displayed_count):
        ax.plot(k_values, spacings[:, i], color=colors[i], linewidth=1.0)
    quotient = int(np.floor(d / n))
    ax.set(xlim=(0, 1), xlabel=r"$k$", ylabel=r"$\Delta X_i$", title=rf"Components of $\Delta X_{{i,{n},{d}}}(k)$")
    ax.set_xticks([0.0, k_c, 1.0], [r"$0$", rf"$\frac{{{n}}}{{{n + d}}}$", r"$1$"])
    ax.set_yticks(
        [quotient / d, 1 / n, (quotient + 1) / d],
        [rf"$\frac{{{quotient}}}{{{d}}}$", rf"$\frac{{1}}{{{n}}}$", rf"$\frac{{{quotient + 1}}}{{{d}}}$"],
    )
    ax.tick_params(axis="both", labelsize=13)
    fig.tight_layout()
    if EXPORT_FIGURES:
        maybe_save(fig, "Spacing Vector.svg" if (n, d) == (53, 67) else f"Spacing-Vector-n{n}-d{d}.svg")
    return fig, {"n": n, "d": d, "k_c": k_c, "k_values": k_values, "spacings": spacings}

spacing_controls = parameterized_plot(
    make_spacing_figure, default_n=53, default_d=67, parity="odd", n_max=61, d_max=89
)

### The endpoint is the Quplet

The quantized endpoint

$$Q(n,d)=\operatorname{Huplet}(n,d,1)$$

comes from nearest-grid indices

$$a_i=\left\lfloor\frac{id}{n}+\frac12\right\rfloor.$$

Its gaps are balanced not only between adjacent onsets but at every cyclic scale: an $\ell$-onset distance is always either $\lfloor \ell d/n\rfloor$ or $\lceil \ell d/n\rceil$. This identifies the gap necklace with the Euclidean rhythm $E(n,d)$.

The Euclidean class forgets rotation and labels; the dynamics retain both through the fixed phase $X_0=0$ and the labels inherited from the initial grid. This is why the paper calls $Q(n,d)$ a **canonical representative**, not a new Euclidean-rhythm class.

The threshold and endpoint are linked by the exact identity

$$\operatorname{Huplet}(n,d,k_c)=Q(n,n+d).$$

In [108]:
def quplet_indices_and_gaps(n, d):
    indices = np.array([nearest_integer(i * d / n) for i in range(n)])
    gaps = np.diff(np.r_[indices, d])
    return indices, gaps

def quplet_summary_html(n, d):
    threshold_indices, threshold_gaps = quplet_indices_and_gaps(n, n + d)
    endpoint_indices, endpoint_gaps = quplet_indices_and_gaps(n, d)
    return (
        f"<b>Q({n},{n+d})</b> — balanced grid: indices {threshold_indices.tolist()}, gaps {threshold_gaps.tolist()} / {n+d}<br>"
        f"<b>Q({n},{d})</b> — endpoint grid: indices {endpoint_indices.tolist()}, gaps {endpoint_gaps.tolist()} / {d}"
    )

if WIDGETS_AVAILABLE:
    quplet_summary = HTML()

    def update_quplet_summary(change=None):
        state = crest_controls["state"]
        quplet_summary.value = quplet_summary_html(state["n"], state["d"]) if state else "Choose a valid pair above."

    for control in (crest_controls["n_slider"], crest_controls["d_slider"]):
        control.observe(update_quplet_summary, names="value")
    update_quplet_summary()
    display(quplet_summary)
else:
    print(quplet_summary_html(5, 7))

HTML(value='<b>Q(5,12)</b> — balanced grid: indices [0, 2, 5, 7, 10], gaps [2, 3, 2, 3, 2] / 12<br><b>Q(5,7)</…

## 5. Why these crests? Full-interval prominence

Continuation labels $n$ branches, but the slider showed that other crests may appear. A dynamical selection mechanism is convincing only if its labeled branches remain perceptually distinguished.

The paper proves that for every $0<k<1$, the anchored branches are **exactly the $n$ highest local maxima** of $F_k$. At $k=1$ all $d$ crests tie, so strict ranking necessarily ends there.

For branch $i$, define its height

$$A_i(k)=F_k(X_i(k)).$$

In the odd case, reflected branches have equal heights, so the plot displays $A_0$ and one member of each reflected pair. Every displayed curve reaches its minimum at the selected pair's balanced value $k_c=n/(n+d)$. The controls open at the paper's example $(5,7)$.

In [109]:
def make_amplitude_figure(n, d):
    k_c = n / (n + d)
    k_values = parameter_grid(k_c, points=501)
    positions = anchored_branches(n, d, k_values)
    amplitudes = F(positions.T, k_values, n, d)
    critical_index = np.flatnonzero(np.isclose(k_values, k_c))[0]
    class_count = (n + 1) // 2
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    colors = plt.cm.viridis(np.linspace(0.0, 1.0, class_count))
    for i in range(class_count):
        ax.plot(k_values, amplitudes[i], color=colors[i], linewidth=1.8, label=rf"$A_{i}(k)$")
        if i > 0:
            ax.scatter([k_c], [amplitudes[i, critical_index]], color=colors[i], s=24, zorder=4)
    ax.axvline(k_c, color="black", linestyle="--", linewidth=0.9)
    ax.annotate(r"$k_c$", xy=(k_c, 0.0), xycoords=ax.get_xaxis_transform(), xytext=(0, -3), textcoords="offset points", ha="center", va="top", annotation_clip=False)
    ax.set(xlim=(0, 1), xlabel=r"$k$", ylabel=r"$A_i(k)$", title=rf"Anchored amplitudes: $(n,d)=({n},{d})$")
    ax.set_xticks([0.0, 1.0], [r"$0.0$", r"$1.0$"])
    ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.55))
    fig.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
    if EXPORT_FIGURES:
        maybe_save(fig, "amplitude-trajectories-n5-d7.svg" if (n, d) == (5, 7) else f"amplitude-trajectories-n{n}-d{d}.svg")
    return fig, {"n": n, "d": d, "k_c": k_c, "k_values": k_values, "amplitudes": amplitudes}

amplitude_controls = parameterized_plot(
    make_amplitude_figure, default_n=5, default_d=7, parity="odd", n_max=19, d_max=41
)

This prominence result closes the selection chain: the branches are not merely mathematically traceable; they are the top-$n$ crest selection throughout the open deformation interval.

## 6. Even $n$: one rounding tie becomes a pitchfork

Odd $n$ makes nearest-grid selection unique. When $n$ is even, the central anchor $x=1/2$ is exactly tied between two reflected $d$-grid sites. The dynamics reflect that arithmetic ambiguity.

For even $n$ and odd $d$, the central crest loses stability at

$$k_*=\frac{n^2}{n^2+d^2}.$$

It splits into two reflected crest arms ending at $(d-1)/(2d)$ and $(d+1)/(2d)$, producing two equally canonical reflected Quplets. The controls open at the paper's example $(4,7)$; they require even $n$ and a coprime larger $d$ (which is then automatically odd). A signed infinitesimal perturbation can select one arm, as in the original `crests.py` pitchfork calculation.

In [110]:
def central_pitchfork_arms(n, d, k_values):
    if n % 2 or d % 2 != 1:
        raise ValueError("This central pitchfork requires even n and odd d.")

    center = 0.5
    k_star = n**2 / (n**2 + d**2)
    left_endpoint = (d - 1) / (2 * d)
    right_endpoint = (d + 1) / (2 * d)
    left_arm = np.full_like(k_values, center, dtype=float)
    right_arm = np.full_like(k_values, center, dtype=float)

    def deflated_stationarity(x, k):
        if np.isclose(x, center, atol=1e-14):
            return 2 * np.pi * ((1 - k) * n**2 - k * d**2)
        return G(x, k, n, d) / (x - center)

    for j, k in enumerate(k_values):
        if k <= k_star:
            continue
        if k >= 1.0:
            left_arm[j], right_arm[j] = left_endpoint, right_endpoint
            continue
        left_arm[j] = brentq(deflated_stationarity, left_endpoint, center, args=(k,), xtol=2e-14, rtol=1e-14)
        right_arm[j] = brentq(deflated_stationarity, center, right_endpoint, args=(k,), xtol=2e-14, rtol=1e-14)

    return k_star, left_arm, right_arm


def make_pitchfork_figure(n, d):
    k_star = n**2 / (n**2 + d**2)
    k_values = parameter_grid(k_star, points=501)
    noncentral = anchored_branches(n, d, k_values, skip_indices={n // 2})
    k_star, left_arm, right_arm = central_pitchfork_arms(n, d, k_values)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.0, 1.0, n + 1))
    for i in range(n + 1):
        if i != n // 2:
            ax.plot(k_values, noncentral[:, i], color=colors[i], linewidth=1.6)
    ax.plot(k_values, left_arm, color=colors[n // 2], linewidth=1.6)
    ax.plot(k_values, right_arm, color=colors[n // 2], linewidth=1.6)
    ax.axvline(k_star, color="gray", linewidth=1.0, linestyle="--")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=r"$k$", ylabel=r"$X_i$", title=rf"$X_{{i,{n},{d}}}(k)$")
    ax.set_xticks([0.0, k_star, 1.0], [r"$0$", rf"$\frac{{{n**2}}}{{{n**2 + d**2}}}$", r"$1$"])
    left_ticks, left_labels = fraction_ticks(n)
    ax.set_yticks(left_ticks, left_labels)
    right_ax = ax.secondary_yaxis("right")
    right_ticks, right_labels = fraction_ticks(d)
    right_ax.set_yticks(right_ticks, right_labels)
    ax.tick_params(axis="both", labelsize=12)
    right_ax.tick_params(axis="y", labelsize=12)
    fig.tight_layout()
    if EXPORT_FIGURES:
        maybe_save(fig, "pitchfork.svg" if (n, d) == (4, 7) else f"pitchfork-n{n}-d{d}.svg")
    return fig, {"n": n, "d": d, "k_star": k_star, "k_values": k_values, "left_arm": left_arm, "right_arm": right_arm}

pitchfork_controls = parameterized_plot(
    make_pitchfork_figure, default_n=4, default_d=7, parity="even", n_max=20, d_max=41, n_step=2
)

## 7. The paper's argument in one view

**Continuous signal**

$$F_k \quad\longrightarrow\quad \text{anchored highest crests }X_i(k)$$

**Continuous rhythm**

$$X_i(k) \quad\longrightarrow\quad \operatorname{Huplet}(n,d,k)=(X_{i+1}-X_i)_i$$

**Two exact quantizations**

$$\operatorname{Huplet}(n,d,k_c)=Q(n,n+d), \qquad \operatorname{Huplet}(n,d,1)=Q(n,d)$$

**Discrete interpretation**

$$Q(n,d)=\text{the anchored nearest-grid representative of }E(n,d).$$

For odd $n$, this representative is unique. For even $n$, the single central tie becomes a pitchfork and yields a reflected canonical pair.

## Reproducibility checks

These numerical assertions check the exact grid-locking values emphasized above. They are consistency checks for the notebook, not substitutes for the paper's proofs.

In [111]:
threshold_state = threshold_controls["state"]
spacing_state = spacing_controls["state"]
pitchfork_state = pitchfork_controls["state"]
assert threshold_state and spacing_state and pitchfork_state, "Choose valid parameter pairs before running the checks."

# Selected threshold crests lie on the current (n+d)-grid.
threshold_N = threshold_state["N"]
assert np.allclose(threshold_state["selected_x"] * threshold_N, np.round(threshold_state["selected_x"] * threshold_N))

# The current spacing example has the two nearest-grid values at k_c and k=1.
spacing_n, spacing_d = spacing_state["n"], spacing_state["d"]
spacing_N = spacing_n + spacing_d
spacing_threshold_index = np.flatnonzero(np.isclose(spacing_state["k_values"], spacing_state["k_c"]))[0]
threshold_row = spacing_state["spacings"][spacing_threshold_index]
endpoint_row = spacing_state["spacings"][-1]
expected_threshold = {int(np.floor(spacing_N / spacing_n)), int(np.ceil(spacing_N / spacing_n))}
expected_endpoint = {int(np.floor(spacing_d / spacing_n)), int(np.ceil(spacing_d / spacing_n))}
assert set(np.round(threshold_row * spacing_N).astype(int)) == expected_threshold
assert set(np.round(endpoint_row * spacing_d).astype(int)) == expected_endpoint

# The current reflected pitchfork endpoints are (d-1)/(2d) and (d+1)/(2d).
pitchfork_d = pitchfork_state["d"]
assert np.isclose(pitchfork_state["left_arm"][-1], (pitchfork_d - 1) / (2 * pitchfork_d))
assert np.isclose(pitchfork_state["right_arm"][-1], (pitchfork_d + 1) / (2 * pitchfork_d))

print("All narrative-notebook checks passed.")

All narrative-notebook checks passed.
